# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading, inspecting, and performing basic data analysis on the FAIR² (Mass Spectrometry) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant (JSON-LD schema) available at the following URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and its records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Dataset
dataset = mlc.Dataset(croissant_url)

# Show general metadata overview
print('Dataset loaded.')
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', None)}")
print(f"Date Published: {getattr(dataset.metadata, 'datePublished', None)}")
print(f"License: {getattr(dataset.metadata, 'license', None)}")
print(f"Version: {getattr(dataset.metadata, 'version', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s for inspection.

### List available record sets (referenced by `@id`):

In [ ]:
# List all record sets in the dataset, referencing their '@id'
record_sets = list(dataset.metadata.get_record_sets())
print('Record sets in this dataset:')
for rset in record_sets:
    print(f"@id: {rset['@id']}", f"| name: {rset.get('name', 'N/A')}")

#### For each record set, list available fields (referenced by their field and column `@id`):

In [ ]:
from pprint import pprint
# Display fields (and column ids) for each record set
for rset in record_sets:
    print(f"\nRecord set: {rset['@id']} ({rset.get('name', 'N/A')})")
    fields = rset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  field @id: {field['@id']}  name: {field.get('name', '-')}")
        col = field.get('column', None)
        if col:
            if isinstance(col, list):
                for c in col:
                    print(f"    -> column @id: {c['@id']} name: {c.get('name', '-')}")
            else:
                print(f"    -> column @id: {col['@id']} name: {col.get('name', '-')}")

## 3. Data Extraction
Let's load data from one or more record sets into pandas DataFrames for inspection and analysis.

We'll demonstrate on the primary data table. Please use the above overview section to select the appropriate record set `@id`s.

In [ ]:
# Pick the first record set (main data table) for extraction
main_record_set_id = record_sets[0]['@id']
# You can add additional record set ids as needed
record_set_ids = [main_record_set_id]

dataframes = {}
for rset_id in record_set_ids:
    print(f"Loading records from record set: {rset_id}")
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rset_id)))
        dataframes[rset_id] = df
        print(f"  Columns: {list(df.columns)}\n  Rows: {df.shape[0]}")
    except Exception as e:
        print(f"Could not load data from record set {rset_id}: {e}")

# Preview first rows
df = dataframes[main_record_set_id]
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's explore and process the data. For demonstration, choose a numeric field (e.g., abundance or MW) as referenced by its field `@id` or DataFrame column name as above.

We'll show:
- Filtering for proteins with MW (Molecular Weight) > threshold
- Normalizing the MW field
- Grouping (if any categorical field is present, e.g., by `description` or other)

Update variable names below as needed using field `@id`s (see the data overview and DataFrame columns).

In [ ]:
# Example: Work with main_record_set_id and its DataFrame
df = dataframes[main_record_set_id]

# Try to guess the numeric field for demonstration ('MW', 'abundance', or similar)
possible_numeric_fields = [c for c in df.columns if (('mw' in c.lower()) or ('weight' in c.lower()) or ('abund' in c.lower()) or (df[c].dtype.kind in 'fi'))]
print(f"Fields with 'mw'/'weight'/'abund' or numeric types: {possible_numeric_fields}")

# Use the first detected numeric field, or specify explicitly
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]
print(f"Using field '{numeric_field}' for analysis")

# Filter: Only rows with numeric_field > threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize (z-score) the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} field:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a field (choose the first string/categorical field not equal to numeric_field)
categorical_candidates = [c for c in df.columns if (df[c].dtype == 'O' and c != numeric_field)]
if categorical_candidates:
    group_field = categorical_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean {numeric_field} by '{group_field}':")
    print(grouped_df.head())
else:
    print("No categorical field found for grouping.")

## 5. Visualization
Let's visualize the distribution of the numeric field (`MW`, abundance, etc.), and its normalized version. We'll use seaborn/matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field after filtering
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.show()

# Normalized distribution
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True)
plt.title(f"Distribution of normalized {numeric_field} (filtered)")
plt.xlabel(f"{numeric_field} (z-score normalized)")
plt.show()

# If grouped, show bar plot of group means
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- We loaded the Mass Spectrometry dataset using `mlcroissant`, explored its metadata, record sets, and fields by their `@id`s.
- We extracted a main data table, filtered proteins by a numeric field (e.g., MW or abundance), normalized it, and provided some basic grouping and visualization.
- This template can be adapted for deeper bioinformatics/statistics workflows, and easily extended for multiple record sets or additional data cleaning.

For more advanced analyses, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/).